<a href="https://colab.research.google.com/github/anjicx/CNHypergraph/blob/main/FirstScript.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Loading the data

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np


In [ ]:
# loading the data
diagnosis = pd.read_csv("data/table_diagnoses.csv", sep=";")
stays = pd.read_csv("data/final_one_percent_stays.csv", sep=";")
stays_secondaries = pd.read_csv("data/final_one_percent_stays_secondaries.csv", sep=";")
age = pd.read_csv("data/table_age.csv", sep=";", encoding="latin1")
#merging data
stays = stays.merge(age, on="ag_id", how="left")
stays = stays.rename(columns={"age": "age_group"})

In [ ]:
# patients with >=1 stay before 2003 are removed completely
# then only from 2003 stays are kept

WASHOUT_END   = pd.Timestamp("2002-12-31")

OBS_START = pd.Timestamp("2003-01-01")

stays["entry_date"] = pd.to_datetime(stays["entry_date"], errors="coerce")

# numbers before filtering
n_patients_before = stays["patient_no"].nunique()
n_stays_before = stays.shape[0]

# patients with AT LEAST ONE stay in washout period 1997-2002
patients_pre_2003 = stays.loc[
    (stays["entry_date"] <= WASHOUT_END),
    "patient_no"
].dropna().unique()

# remove these patients completely
stays_after_washout = stays[
    ~stays["patient_no"].isin(patients_pre_2003)
].copy()

# keep only observation period from 2003
stays_obs = stays_after_washout[(stays_after_washout["entry_date"] >= OBS_START) ].copy()

# create summary table
washout_summary = pd.DataFrame([
    {
        "step": "Before filtering",
        "n_patients": n_patients_before,
        "n_stays": n_stays_before,
        "description": "All available patients and stays before applying washout filter"
    },
    {
        "step": "After removing pre-2003 patients",
        "n_patients": stays_after_washout["patient_no"].nunique(),
        "n_stays": stays_after_washout.shape[0],
        "description": "Patients with any stay before 2003 were removed completely"
    },
    {
        "step": "Observation period from 2003",
        "n_patients": stays_obs["patient_no"].nunique(),
        "n_stays": stays_obs.shape[0],
        "description": "Final dataset restricted to stays from 2003 onward"
    }
])

washout_summary.to_csv("washout_summary.csv", index=False)

1. PATIENT FILTERING
Making a cohort of patients who have at least 3 unique years and their first appearance is from 20 age

In [ ]:
#Patient stays filtered


patient_stays = stays_obs[["patient_no", "entry_date", "exit_date", "sex_id", "age_group"]].copy()
patient_stays = patient_stays[patient_stays["sex_id"].isin([1, 2])].copy()

patient_stays["entry_date"] = pd.to_datetime(patient_stays["entry_date"])
patient_stays["exit_date"] = pd.to_datetime(patient_stays["exit_date"])
patient_stays = patient_stays.sort_values(["patient_no", "entry_date"])#patient stays sorted by entry date


#just takes the year from the entry date
patient_stays["year"] = patient_stays["entry_date"].dt.year
#patient_stays
#COUNTNING NUMBER OF UNIQUE YEARS IN VISITS OF EACH PATIENT
unique_years = (
    patient_stays.groupby("patient_no")["year"].nunique()# group by patient, count distinct years
    .reset_index(name="n_unique_years")
)

  # assign age group and sex from the first visit
first_visit = patient_stays.drop_duplicates("patient_no", keep="first")[
    ["patient_no", "sex_id", "age_group"]
]

patient_filter = first_visit.merge(unique_years, on="patient_no", how="left")

patient_filter["sex"] = patient_filter["sex_id"].map({
    1: "Male",
    2: "Female"
})


In [ ]:

# copy patient table before age filtering
age_plot_df = patient_filter.copy()

# extract age_start from age_group
age_plot_df["age_start"] = (
    age_plot_df["age_group"]
    .astype(str)
    .str.extract(r"(\d+)")
    .astype(float)
)

# all diagnoses from 2003+ filtered data
primary_dx = stays_obs[["stay_id", "patient_no", "pri_diag_id"]].rename(
    columns={"pri_diag_id": "diagnose_id"}
)

secondary_dx = stays_secondaries.merge(
    stays_obs[["stay_id", "patient_no"]],
    on="stay_id",
    how="inner"
)[["stay_id", "patient_no", "sec_diag_id"]].rename(
    columns={"sec_diag_id": "diagnose_id"}
)

all_dx = pd.concat([primary_dx, secondary_dx], ignore_index=True)

all_dx["diagnose_id"] = pd.to_numeric(all_dx["diagnose_id"], errors="coerce")
all_dx = all_dx.dropna(subset=["diagnose_id"])

# count unique diagnoses per patient
unique_dx = (
    all_dx
    .drop_duplicates(["patient_no", "stay_id", "diagnose_id"])
    .groupby("patient_no")["diagnose_id"]
    .nunique()
    .reset_index(name="n_unique_diagnoses")
)

age_plot_df = age_plot_df.merge(unique_dx, on="patient_no", how="left")
age_plot_df["n_unique_diagnoses"] = age_plot_df["n_unique_diagnoses"].fillna(0)

# age groups like paper
age_plot_df["age_group_10y"] = pd.cut(
    age_plot_df["age_start"],
    bins=[0, 10, 20, 30, 40, 50, 60, 70, np.inf],
    labels=["0-9", "10-19", "20-29", "30-39", "40-49", "50-59", "60-69", "70+"],
    right=False
)

# percentage of patients with >=3 unique diagnoses
age_burden = (
    age_plot_df
    .groupby("age_group_10y", observed=False)
    .agg(
        n_patients=("patient_no", "nunique"),
        pct_dx_ge_3=("n_unique_diagnoses", lambda x: 100 * (x >= 3).mean())
    )
    .reset_index()
)

age_burden["pct_dx_ge_3"] = age_burden["pct_dx_ge_3"].round(2)

age_burden.to_csv("age_burden_10y_groups.csv", index=False)

age_burden


In [ ]:
plt.figure(figsize=(10, 5))

plt.bar(
    age_burden["age_group_10y"].astype(str),
    age_burden["pct_dx_ge_3"]
)


plt.xlabel("Age group")
plt.ylabel("Patients with ≥3 unique diagnoses (%)")
plt.title("Share of patients with ≥3 unique diagnoses by age group")
plt.tight_layout()

plt.savefig("age_burden_10y_groups.png", dpi=150)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

sex_age_burden = (
    age_plot_df
    .groupby(["age_group_10y", "sex"], observed=False)
    .agg(
        n_patients=("patient_no", "nunique"),
        pct_dx_ge_3=("n_unique_diagnoses", lambda x: 100 * (x >= 3).mean())
    )
    .reset_index()
)

sex_age_burden["pct_dx_ge_3"] = sex_age_burden["pct_dx_ge_3"].round(2)

sex_age_burden_pivot = sex_age_burden.pivot(
    index="age_group_10y",
    columns="sex",
    values="pct_dx_ge_3"
)

sex_age_burden_pivot = sex_age_burden_pivot.reindex([
    "0-9", "10-19", "20-29", "30-39",
    "40-49", "50-59", "60-69", "70+"
])

sex_age_burden_pivot

In [ ]:
x = np.arange(len(sex_age_burden_pivot.index))
width = 0.35

plt.figure(figsize=(11, 5))

plt.bar(
    x - width / 2,
    sex_age_burden_pivot["Female"],
    width,
    label="Female"
)

plt.bar(
    x + width / 2,
    sex_age_burden_pivot["Male"],
    width,
    label="Male"
)

plt.xticks(x, sex_age_burden_pivot.index.astype(str))
plt.xlabel("Age group")
plt.ylabel("Patients with ≥3 unique diagnoses (%)")
plt.title("Average comorbidity burden by age group and sex")
plt.legend()
plt.tight_layout()

plt.savefig("sex_stratified_age_burden_pct_dx_ge_3.png", dpi=150)
plt.show()

In [ ]:

# FIRST CONDITION:20+ AGE GROUP
# age group 20+. Counting first age group in the dataset-to be 20+? 40 bis 44 Jahre lake this is age_group
patient_filter["age_group"] = patient_filter["age_group"].astype(str).str.strip()
patient_filter["age_start"] = (
    patient_filter["age_group"]
    .str.extract(r"(\d+)")
    .astype(float)
)# to extract the start age in the age group
patient_filter = patient_filter[patient_filter["age_start"] >= 20].copy()#filter 20+


In [ ]:
#adding to summary table
age_filter_row = pd.DataFrame([
    {
        "step": "After age filter: age >= 20",
        "n_patients": patient_filter["patient_no"].nunique(),
        "n_stays": stays_obs[
            stays_obs["patient_no"].isin(patient_filter["patient_no"])
        ].shape[0],
        "description": "Patients retained after requiring first observed age group to be at least 20"
    }
])

washout_summary = pd.concat(
    [washout_summary, age_filter_row],
    ignore_index=True
)


In [ ]:

base_cohort = patient_filter.copy()

base_cohort["n_unique_years"] = pd.to_numeric(
    base_cohort["n_unique_years"],
    errors="coerce"
)

base_cohort = base_cohort.dropna(subset=["n_unique_years"]).copy()
base_cohort["n_unique_years"] = base_cohort["n_unique_years"].astype(int)

total_patients = base_cohort["patient_no"].nunique()

max_years = base_cohort["n_unique_years"].max()

retention_rows = []

for threshold in range(1, max_years + 1):
    n_patients = base_cohort.loc[
        base_cohort["n_unique_years"] >= threshold,
        "patient_no"
    ].nunique()
    
    retention_rows.append({
        "min_unique_years": threshold,
        "n_patients": n_patients,
        "percent_retained": 100 * n_patients / total_patients
    })

retention_df = pd.DataFrame(retention_rows)

retention_df.to_csv("unique_years_retention_curve.csv", index=False)

retention_df
plt.figure(figsize=(9, 5))

plt.plot(
    retention_df["min_unique_years"],
    retention_df["percent_retained"],
    marker="o"
)

plt.axvline(2, linestyle="--", label="Possible threshold: >= 2")

for _, row in retention_df.iterrows():
    if row["min_unique_years"] <= 6:
        plt.text(
            row["min_unique_years"],
            row["percent_retained"] + 2,
            f'{row["percent_retained"]:.1f}%',
            ha="center",
            fontsize=9
        )

plt.xlabel("number of unique years")
plt.ylabel("Patients retained (%)")
plt.title("Number of unique years in hospital records")
plt.xticks(retention_df["min_unique_years"])
plt.ylim(0, 105)
plt.grid(axis="y", alpha=0.3)
plt.legend()
plt.tight_layout()

plt.savefig("unique_yearsGraph1.png", dpi=150)
plt.show()

In [ ]:
#histogram number of unique years in hospital records for patients above 20, who dont have any visit before 2003
plt.figure(figsize=(8, 5))

plt.hist(
    patient_filter["n_unique_years"].dropna(),
    bins=range(
        int(patient_filter["n_unique_years"].min()),
        int(patient_filter["n_unique_years"].max()) + 2
    ),
    edgecolor="black"
)

plt.xlabel("Number of unique years")
plt.ylabel("Number of patients")
plt.title("Number of unique years in hospital records (older then 20)")
plt.legend()
plt.tight_layout()

plt.savefig("hist_unique_years_age20.png", dpi=150)
plt.show()

In [ ]:

# SECOND CONDITION NUMBER OF UNIQUE YEARS FOR THAT PATIENT MORE THEN 3
patient_filter = patient_filter[patient_filter["n_unique_years"] >= 3].copy()
#ALL THE PATIENTS THAT CAME THROUGH FILTER

unique_years_row = pd.DataFrame([
    {
        "step": "After unique years filter: n_unique_years >= 3",
        "n_patients": patient_filter["patient_no"].nunique(),
        "n_stays": stays_obs[
            stays_obs["patient_no"].isin(patient_filter["patient_no"])
        ].shape[0],
        "description": "Patients retained after requiring at least 3 unique observation years"
    }
])

washout_summary = pd.concat(
    [washout_summary, unique_years_row],
    ignore_index=True
)

# update removed and percent columns
washout_summary["removed_from_previous"] = (
    washout_summary["n_patients"].shift(1) - washout_summary["n_patients"]
).fillna(0).astype(int)

washout_summary["percent_remaining"] = (
    100 * washout_summary["n_patients"] / washout_summary.loc[0, "n_patients"]
).round(2)

# save updated table
washout_summary.to_csv("washout_summary.csv", index=False)


patient_filter is our cohort of patients -> filtering only the stays from our cohort

In [ ]:
filtered_stays = stays_obs[stays_obs["patient_no"].isin(patient_filter["patient_no"])].copy()
filtered_stays["entry_date"] = pd.to_datetime(filtered_stays["entry_date"])
filtered_stays["exit_date"] = pd.to_datetime(filtered_stays["exit_date"], errors="coerce")

print("Filtered stays shape:", filtered_stays.shape)

Filtered stays shape: (264533, 13)


2. Diagnosis nodes

Creating nodes of diagnosis(dictionary containing all diagnosis), and table of stays and connected diagnosis to that stay.Each row is one diagnosis connected to that stay

In [ ]:
# building nodes of diagnosis

nodes = diagnosis.copy()
nodes["diagnose_id"] = pd.to_numeric(nodes["diagnose_id"])
nodes["node_id"] = range(len(nodes))
nodes = nodes[["node_id", "diagnose_id", "descr", "icd_code"]].copy()



#creating a tbl of primary diagnosis

primary = filtered_stays[["stay_id", "patient_no", "entry_date", "pri_diag_id"]].copy()#stays has for key to diagnosis
primary = primary.rename(columns={"pri_diag_id": "diagnose_id"})#rename foreign key diagnose_id
primary["diagnose_id"] = pd.to_numeric(primary["diagnose_id"])
primary["role"] = "primary"



#creating a tbl of secondary diagnosis

filtered_stays_secondaries = stays_secondaries[stays_secondaries["stay_id"].isin(filtered_stays["stay_id"])].copy()
filtered_stays_secondaries["sec_diag_id"] = pd.to_numeric(filtered_stays_secondaries["sec_diag_id"])
secondary = filtered_stays_secondaries.merge(filtered_stays[["stay_id", "patient_no", "entry_date"]],on="stay_id",how="inner")
secondary = secondary.rename(columns={"sec_diag_id": "diagnose_id"})
secondary["role"] = "secondary"



# NODES TABLE: node id connected to diagnosis
stay_diagnoses = pd.concat([primary, secondary], ignore_index=True)
# remove rows with missing diagnosis ids
stay_diagnoses = stay_diagnoses.dropna(subset=["diagnose_id"]).copy()
# inner join
stay_diagnoses = pd.merge(stay_diagnoses,nodes,on="diagnose_id",how="inner")

#stay_diagnoses.head(5)


,stay_id,patient_no,entry_date,diagnose_id,role,node_id,descr,icd_code
0,4005,16,2015-07-08,1281,primary,1079,BösartigeNeubildung-Sonstigeundnichtnäherbezei...,C725
1,4100,17,2015-11-12,626,primary,516,"Kandidose,nichtnäherbezeichnet,Sooro.n.A.",B379
2,4665,19,2015-04-10,346,primary,319,SonstigenäherbezeichneteSpirochäteninfektionen,A698
3,4708,20,2015-03-26,1458,primary,1250,MelanomainsituansonstigenLokalisationen,D038
4,4904,21,2015-09-17,203,primary,215,Meningokokkenmeningitis(G01*),A390


For each patient keep only the first appearance of diagnosis.Then merge the whole data and sort so you can track which diagnosis was first.Afterwards divide by gender.

In [ ]:
patient_first_diag = (stay_diagnoses.groupby(["patient_no", "node_id"], as_index=False)["entry_date"].min()
.rename(columns={"entry_date": "first_date"})
)
patient_first_diag = patient_first_diag.merge(
    nodes[["node_id", "diagnose_id", "descr", "icd_code"]].drop_duplicates(subset=["node_id"]),
    on="node_id",
    how="left")
# sort for each patient diagnosis order
patient_first_diag = patient_first_diag.sort_values(
    ["patient_no", "first_date", "node_id"]
).copy()


In [ ]:
patient_info = patient_filter[
    ["patient_no", "sex_id", "sex", "age_group"]
].drop_duplicates().copy()

patient_first_diag_sex_age = patient_first_diag.merge(
    patient_info,
    on="patient_no",
    how="left"
)

# sorted for diagnosis timestamp order, unique diagnosis per gender
male_diag = patient_first_diag_sex_age[
    patient_first_diag_sex_age["sex"] == "Male"
].copy()

female_diag = patient_first_diag_sex_age[
    patient_first_diag_sex_age["sex"] == "Female"
].copy()

print("Male rows:", male_diag.shape)
print("Female rows:", female_diag.shape)
male_diag.head(5)

Male rows: (117214, 9)
Female rows: (133788, 9)


,patient_no,node_id,first_date,diagnose_id,descr,icd_code,sex_id,sex,age_group
0,9,1325,2015-01-13,1545,GutartigeNeubildung-KurzeKnochenundGelenkknorp...,D161,1,Male,45 bis 49 Jahre
1,9,1974,2015-11-30,2296,"Hyperlipidämie,nichtnäherbezeichnet",E785,1,Male,45 bis 49 Jahre
2,9,3178,2015-11-30,3703,"Essentielle(primäre)Hypertonie,Bluthochdruck,H...",I10,1,Male,45 bis 49 Jahre
3,9,3226,2015-11-30,3761,"ChronischeischämischeHerzkrankheit,nichtnäherb...",I259,1,Male,45 bis 49 Jahre
4,9,3682,2015-11-30,4279,"ChronischeobstruktiveLungenkrankheit,nichtnähe...",J449,1,Male,45 bis 49 Jahre


Instead of incidence matrix the hyperedge table is sotred, where for each patient from the previous table we add hyperedge index.

In [ ]:
def build_hypergraph_tables(df):
    df = df.sort_values(["patient_no", "first_date", "node_id"]).copy()

    # One hyperedge = one patient
    df["hyperedge_index"] = df["patient_no"]

    # Hyperedge table: patient-diagnosis memberships
    hyperedge_table = df[
        [
            "hyperedge_index",
            "patient_no",
            "sex_id",
            "sex",
            "age_group",
            "node_id",
            "diagnose_id",
            "descr",
            "icd_code",
            "first_date"
        ]
    ].copy()

    # One row per patient with diagnosis sequence
    patient_sequences = (
        df.groupby(["patient_no", "hyperedge_index", "sex_id", "sex", "age_group"])
        .agg(
            sequence=("node_id", list),
            icd_sequence=("icd_code", list),
            first_date_sequence=("first_date", list),
            n_unique_diagnoses=("node_id", "nunique")
        )
        .reset_index()
    )

    return hyperedge_table, patient_sequences


In [ ]:
male_hyperedge_table, male_sequences = build_hypergraph_tables(male_diag)
female_hyperedge_table, female_sequences = build_hypergraph_tables(female_diag)

In [ ]:
#male_hyperedge_table.head(5)
#male_sequences.head(5)

SAVING PART

In [ ]:
male_hyperedge_table.to_csv("male_hyperedge_table.csv", index=False)
female_hyperedge_table.to_csv("female_hyperedge_table.csv", index=False)
male_sequences.to_csv("male_sequences.csv", index=False)
female_sequences.to_csv("female_sequences.csv", index=False)


Saved male and female hypergraph tables to: /content/drive/MyDrive/PatientData/saved_hypergraphs
